# Name That LLM: TF-IDF + stylometry LinearSVC with a style-matched Open Pool sample

Generated by `name_that_llm/kaggle_notebook.py`; do not edit by hand.

* **Model:** character 1-5-grams + word 1-2-grams + 16 punctuation/style rates, then LinearSVC.
* **Data:** `train.csv` plus 2,000 passages per model from the public [Open Pool](https://www.kaggle.com/datasets/crtproduction/name-that-llm-open-pool), sampled by its `match_weight` so typography and length match the real passages.
* **Post-processing:** partial Sinkhorn class balancing over the test set.
* **Local grouped CV** (5 folds by `prompt_id`): balanced accuracy 0.451.

Needs no GPU and no internet; trains in about 15 minutes.

In [ ]:
import glob
import os
import re
import time

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

ON_KAGGLE = os.path.exists('/kaggle/input')
if ON_KAGGLE:
    COMP = next(p for p in ['/kaggle/input/competitions/name-that-llm', '/kaggle/input/name-that-llm'] if os.path.exists(p))
    pools = glob.glob('/kaggle/input/**/ntl_open_pool.parquet', recursive=True)
    assert pools, 'Attach the Open Pool dataset (crtproduction/name-that-llm-open-pool)'
    POOL = pools[0]
    OUT = 'submission.parquet'
else:  # running from this project's notebooks/ folder
    COMP = '../data/raw'
    POOL = '../data/external/open_pool/ntl_open_pool.parquet'
    OUT = '../data/processed/submission_from_notebook.parquet'

# settings of the committed local model (reports/pool2000_w1_cv_metrics.json)
N_PER_LABEL, POOL_SEED, POOL_WEIGHT = 2000, 0, 1.0
C, STYLE_WEIGHT = 0.5, 0.2
BALANCE_TEMPERATURE, BALANCE_STRENGTH = 1.0, 0.3
print('competition data:', COMP)
print('open pool:', POOL)

## Features (copied from `name_that_llm/features.py`)

In [ ]:
import re

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import StandardScaler

# name -> regex; counted per 100 words
PUNCT_PATTERNS = {
    "em_dash": "—",
    "en_dash": "–",
    "semicolon": ";",
    "colon": ":",
    "comma": ",",
    "question": r"\?",
    "exclaim": "!",
    "parens": r"\(",
    "curly_quote": "[‘’“”]",
    "straight_quote": "['\"]",
    "digits": r"\d",
}
_SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")
_WORD = re.compile(r"[a-z']+")


def style_features(texts) -> pd.DataFrame:
    """Dense stylometric features: punctuation rates, length and lexical diversity."""
    s = pd.Series(list(texts), dtype=object)
    words = s.str.split().str.len().clip(lower=1)
    out = {name: s.str.count(pat) / words * 100 for name, pat in PUNCT_PATTERNS.items()}
    out["has_em_dash"] = s.str.contains("—").astype(float)
    out["has_curly"] = s.str.contains("[‘’“”]").astype(float)
    out["log_words"] = np.log(words)
    sentences = s.map(lambda t: max(1, len([x for x in _SENTENCE_SPLIT.split(t.strip()) if x])))
    out["words_per_sentence"] = words / sentences
    tokens = s.str.lower().map(_WORD.findall)
    out["avg_word_len"] = tokens.map(lambda w: np.mean([len(x) for x in w]) if w else 0.0)
    out["type_token"] = tokens.map(lambda w: len(set(w)) / max(1, len(w)))
    return pd.DataFrame(out).fillna(0.0)


class StyleFeatures(BaseEstimator, TransformerMixin):
    """sklearn wrapper around :func:`style_features`."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return style_features(X).to_numpy(dtype=float)

    def get_feature_names_out(self, input_features=None):
        return np.array(style_features(["a b."]).columns, dtype=object)


def build_features(style_weight: float = 1.0) -> FeatureUnion:
    """Character n-grams + word n-grams + scaled stylometric features."""
    return FeatureUnion(
        [
            (
                "char",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(1, 5),
                    lowercase=False,
                    sublinear_tf=True,
                    min_df=2,
                    max_features=300_000,
                    dtype=np.float32,
                ),
            ),
            (
                "word",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    lowercase=True,
                    token_pattern=r"(?u)\b\w[\w']*\b",
                    sublinear_tf=True,
                    min_df=2,
                    dtype=np.float32,
                ),
            ),
            ("style", make_pipeline(StyleFeatures(), StandardScaler())),
        ],
        transformer_weights={"char": 1.0, "word": 1.0, "style": style_weight},
    )


## Class balancing (copied from `name_that_llm/modeling/postprocess.py`)

In [ ]:
def softmax(scores: np.ndarray, temperature: float) -> np.ndarray:
    z = scores * temperature
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def sinkhorn(proba: np.ndarray, strength: float = 1.0, n_iter: int = 50) -> np.ndarray:
    """Geometric mix of ``proba`` and its column-balanced Sinkhorn projection."""
    if strength <= 0:
        return proba
    n, k = proba.shape
    q = proba + 1e-12
    for _ in range(n_iter):
        q = q / q.sum(axis=0, keepdims=True) * (n / k)  # each class gets n/k total mass
        q = q / q.sum(axis=1, keepdims=True)  # each row is a distribution
    logp = (1 - strength) * np.log(proba + 1e-12) + strength * np.log(q)
    p = np.exp(logp - logp.max(axis=1, keepdims=True))
    return p / p.sum(axis=1, keepdims=True)


## Data: train.csv + a style-matched Open Pool sample

In [ ]:
train = pd.read_csv(f'{COMP}/train.csv')
test = pd.read_csv(f'{COMP}/test.csv')
pool = pd.read_parquet(POOL, columns=['text', 'model', 'match_weight'])

# same draw as name_that_llm/dataset.py: weighted sampling without replacement
# (Efraimidis-Spirakis) by match_weight, the same number of passages per label
parts = []
for label, d in pool.groupby('model'):
    rng = np.random.default_rng([POOL_SEED, len(parts)])
    keys = rng.random(len(d)) ** (1.0 / d.match_weight.clip(lower=1e-6).to_numpy())
    parts.append(d.iloc[np.argsort(-keys)[:min(N_PER_LABEL, len(d))]])
pool_sample = pd.concat(parts, ignore_index=True)

LABELS = sorted(train.model.unique())
assert len(LABELS) == 18 and set(pool_sample.model) <= set(LABELS)
print(f'train {len(train)}, pool sample {len(pool_sample)}, test {len(test)}')

## Train

In [ ]:
X = pd.concat([train.text, pool_sample.text], ignore_index=True)
y = np.concatenate([train.model.to_numpy(), pool_sample.model.to_numpy()])
w = np.concatenate([np.ones(len(train)), np.full(len(pool_sample), POOL_WEIGHT)])

model = Pipeline([
    ('features', build_features(STYLE_WEIGHT)),
    ('clf', LinearSVC(C=C, class_weight='balanced', max_iter=5000)),
])
t0 = time.time()
model.fit(X, y, clf__sample_weight=w)
print(f'trained in {time.time() - t0:.0f}s; in-sample balanced acc on train.csv: '
      f'{np.mean([np.mean(model.predict(train.text[train.model == m]) == m) for m in LABELS]):.3f}')

## Predict, balance, write `submission.parquet`

In [ ]:
proba = softmax(model.decision_function(test.text), BALANCE_TEMPERATURE)
pred = np.asarray(model.classes_)[sinkhorn(proba, BALANCE_STRENGTH).argmax(axis=1)]
submission = pd.DataFrame({'id': test.id, 'model': pred})

assert submission.id.is_unique and len(submission) == len(test)
assert submission.model.notna().all() and submission.model.isin(LABELS).all()
submission.to_parquet(OUT, index=False)
print(submission.model.value_counts().to_string())
submission.head()